# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/h0pi/flyrank-ml-intenrship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is content-performance prioritisation. The task is a ranking problem: given a set of content pages, rank them by which pages the content team should review first.

The output is a priority score, not an automatic decision to change or remove a page. The top-ranked pages will be reviewed by a human who can decide whether to refresh content, investigate a technical issue, improve internal linking, or leave the page unchanged.

Ranking fits better than simple classification because the team has limited time and needs an ordered review queue, not only a yes/no label.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The starter-data proxy target is `is_declining_proxy`.

It is created from the observed `trend_direction` field:
- `1` when `trend_direction` is `down`
- `0` when the trend is `stable` or `up`

This is a defined rule based on an observed directional trend, not a final business-outcome label. It is useful for framing and testing the workflow, but a stronger future target would measure whether a page has a meaningful decline in traffic, conversions, or another agreed content-performance outcome over a fixed future time window.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

I will use Precision@50 as the primary success metric.

Precision@50 is the share of the 50 highest-priority pages that are actually declining according to the proxy target.

A starting target of Precision@50 >= 0.70 would mean that at least 35 of the first 50 pages reviewed are genuinely in the declining group. This is useful because the content team can act on a limited weekly review queue.

I would also compare the result with a simple fixed-rule baseline, such as reviewing pages with `trend_direction == "down"` and low recent traffic, before claiming that an ML approach adds value.

In [16]:
# Precision@50 target for the later ranking evaluation
k = 50
target_precision_at_k = 0.70

print(f"Primary metric: Precision@{k}")
print(f"Starting definition of good: >= {target_precision_at_k:.0%}")
print(f"That means at least {int(k * target_precision_at_k)} of the top {k} pages are declining.")

Primary metric: Precision@50
Starting definition of good: >= 70%
That means at least 35 of the top 50 pages are declining.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row represents one anonymised content page at the measurement snapshot used in the starter data.

Each row contains page-level signals that may help prioritise review, such as traffic or engagement measures, content attributes, and a directional trend field. The data is anonymised: no client names, URLs, or private search queries are used.

The dataframe below loads the content lane slice, creates the proxy target, and displays the available columns and a small sample of rows.

In [17]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

# Defined proxy target:
# 1 = the observed directional trend is down
# 0 = stable or up
df["is_declining_proxy"] = (
    df["trend_direction"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("down")
    .astype(int)
)

print(f"Loaded: {DATA_PATH}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("Unit of analysis: one row = one anonymised content page.")
print("\nTrend-direction distribution:")
display(
    df["trend_direction"]
    .value_counts(dropna=False)
    .rename_axis("trend_direction")
    .reset_index(name="page_count")
)

print("\nProxy-target distribution:")
display(
    df["is_declining_proxy"]
    .value_counts(dropna=False)
    .rename_axis("is_declining_proxy")
    .reset_index(name="page_count")
)

print("\nExample rows, including the proxy target:")
display(
    df[
        [
            "content_id",
            "content_type",
            "main_intent",
            "impressions_90d",
            "clicks_90d",
            "sessions_90d",
            "trend_direction",
            "trend_pct",
            "is_declining_proxy",
        ]
    ].head(10)
)

Loaded: data/raw/content_refresh_anonymized.csv
Rows: 30,000
Columns: 45
Unit of analysis: one row = one anonymised content page.

Trend-direction distribution:


,trend_direction,page_count
0,down,16262
1,stable,5962
2,up,4388
3,new,2236
4,flat,1152



Proxy-target distribution:


,is_declining_proxy,page_count
0,1,16262
1,0,13738



Example rows, including the proxy target:


,content_id,content_type,main_intent,impressions_90d,clicks_90d,sessions_90d,trend_direction,trend_pct,is_declining_proxy
0,content_304f48230142,keyword article,transactional,3803,29,17,down,-41.4,1
1,content_a1fb4e703a9e,keyword article,informational,15320,7,9,down,-57.7,1
2,content_9aa793d4d895,keyword article,informational,12581,11,11,down,-60.9,1
3,content_331d6c4de07b,keyword article,commercial,11751,58,78,stable,-13.8,0
4,content_d99b7a2d90ca,keyword article,informational,19140,24,145,down,-34.7,1
5,content_d4084a4bc775,keyword article,transactional,3970,1,5,down,-38.9,1
6,content_9a34b442b552,keyword article,informational,20,0,1,down,-92.3,1
7,content_a63219c6e95a,keyword article,commercial,1724,1,28,stable,0.6,0
8,content_5e6c160719bc,keyword article,informational,32574,29,68,down,-58.8,1
9,content_c27558df2b0c,keyword article,informational,1240,2,3,down,-29.2,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can flag pages with a downward trend, but it cannot easily decide which of many declining pages deserves attention first.

The priority depends on several signals that can interact: the direction and size of change, current traffic level, engagement or conversion signals, page age, content characteristics, and possible data noise. For example, a small decline on a high-traffic page may be more important than a large decline on a low-traffic page.

A ranking model can learn how these signals combine and produce a consistent review order. It remains decision-support, not automatic action: a human reviews the evidence before making content changes.

A future ranking model should be compared with this transparent rule-based baseline on Precision@50. If it does not improve the review queue, the simpler rule should be preferred.

In [18]:
# Transparent rule baseline:
# give highest priority to pages with an observed downward trend,
# then put larger declines first.

baseline_queue = (
    df.assign(
        baseline_priority=df["is_declining_proxy"],
        decline_size=(-df["trend_pct"]).clip(lower=0)
    )
    .sort_values(
        by=["baseline_priority", "decline_size"],
        ascending=[False, False]
    )
    .head(50)
)

baseline_precision_at_50 = baseline_queue["is_declining_proxy"].mean()

print("Baseline rule: prioritise pages with trend_direction == 'down',")
print("then rank them by the size of the observed decline (trend_pct).")
print(f"Baseline Precision@50: {baseline_precision_at_50:.1%}")

display(
    baseline_queue[
        [
            "content_id",
            "trend_direction",
            "trend_pct",
            "is_declining_proxy",
            "baseline_priority",
        ]
    ].head(10)
)

Baseline rule: prioritise pages with trend_direction == 'down',
then rank them by the size of the observed decline (trend_pct).
Baseline Precision@50: 100.0%


,content_id,trend_direction,trend_pct,is_declining_proxy,baseline_priority
23,content_2da6ae9d0882,down,-100.0,1,1
39,content_4595e8704e07,down,-100.0,1,1
48,content_326fa2fa449f,down,-100.0,1,1
49,content_f0717373e86e,down,-100.0,1,1
51,content_d8a23b5e10c5,down,-100.0,1,1
67,content_9042a5355ff7,down,-100.0,1,1
68,content_6bc2ec5f6061,down,-100.0,1,1
111,content_46eaff5b4ae8,down,-100.0,1,1
130,content_27682a7cd588,down,-100.0,1,1
141,content_0af426466565,down,-100.0,1,1


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.